<a href="https://colab.research.google.com/github/ltphongssvn/cs1090b_HallucinationLegalRAGChatbots/blob/feature%2Fdata-acquisition/Making_the_Most_of_your_Colab_Subscription.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CS1090B — Hallucination in Legal RAG Chatbots — End-to-End Pipeline

Runs the data pipeline described in the README on Google Colab Pro A100.
Stages: bootstrap → env check → CourtListener ingest → data audit → dataset probe → LePaRD ingest → LePaRD↔CL compat audit.

In [ ]:
# Cell 2: Bootstrap repo + uv venv (pipeline cells will subprocess via .venv/bin/python)
import os
import pathlib
import subprocess

REPO_URL = "https://github.com/ltphongssvn/cs1090b_HallucinationLegalRAGChatbots.git"
REPO_DIR = "/content/cs1090b_HallucinationLegalRAGChatbots"

def sh(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, text=True)
    if check and r.returncode != 0:
        raise SystemExit(f"command failed: {cmd}")


if not pathlib.Path(REPO_DIR).exists():
    sh(f"git clone {REPO_URL} {REPO_DIR}")

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

if subprocess.run("command -v uv", shell=True).returncode != 0:
    sh("curl -LsSf https://astral.sh/uv/install.sh | sh")
    os.environ["PATH"] = f"/root/.local/bin:{os.environ['PATH']}"

sh("uv python install 3.11.9")
sh("uv python pin 3.11.9")
if not pathlib.Path(".venv").exists():
    sh("uv sync")

pathlib.Path(".env").write_text(
    "export PYTHONHASHSEED=0\n"
    "export CUBLAS_WORKSPACE_CONFIG=:4096:8\n"
    "export TOKENIZERS_PARALLELISM=false\n"
    "export RANDOM_SEED=0\n"
    "export TARGET_GPU_COUNT=1\n"
    "export TARGET_VRAM_GB_MIN=20.0\n"
)

# Verify .venv has correct pinned versions
r = subprocess.run(
    [
        ".venv/bin/python",
        "-c",
        "import sys, numpy, torch, transformers; "
        "print(f'python {sys.version.split()[0]}'); "
        "print(f'numpy {numpy.__version__}'); "
        "print(f'torch {torch.__version__}'); "
        "print(f'transformers {transformers.__version__}')",
    ],
    capture_output=True,
    text=True,
)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr)
    raise SystemExit("venv verification failed")

$ git clone https://github.com/ltphongssvn/cs1090b_HallucinationLegalRAGChatbots.git /content/cs1090b_HallucinationLegalRAGChatbots
cwd: /content/cs1090b_HallucinationLegalRAGChatbots
$ uv python install 3.11.9
$ uv python pin 3.11.9
$ uv sync
python 3.11.9
numpy 1.26.4
torch 2.0.1+cu117
transformers 4.41.2



# Login with a web browser, at Terminal prompt
git config --global user.email "ltphongssvn@gmail.com" && git config --global user.name "ltphongssvn"


gh auth login --hostname github.com --git-protocol https --web

In [4]:
# Cell 3: Environment verification — runs notebooks/cs1090b_HallucinationLegalRAGChatbots.md Cell 1
import os
import pathlib
import re
import subprocess
import sys

os.chdir("/content/cs1090b_HallucinationLegalRAGChatbots")

nb_src = pathlib.Path("notebooks/cs1090b_HallucinationLegalRAGChatbots.md").read_text()
blocks = re.findall(r"```\{code-cell\}[^\n]*\n(.*?)```", nb_src, flags=re.DOTALL)
assert blocks, "no code cells found in notebook md"
cell1_code = blocks[0].replace("os.chdir(os.path.dirname(os.getcwd()))", "# chdir not needed — already at repo root")

env = {**os.environ, "PYTHONUNBUFFERED": "1"}
proc = subprocess.Popen(
    [".venv/bin/python", "-u", "-c", cell1_code],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
    text=True,
    env=env,
)
for line in proc.stdout:
    sys.stdout.write(line)
    sys.stdout.flush()
proc.wait()
if proc.returncode != 0:
    raise SystemExit(f"Cell 3 failed with exit code {proc.returncode}")

  [repro] Reproducibility configured:
    PYTHONHASHSEED=0
    CUBLAS_WORKSPACE_CONFIG=:4096:8
    TOKENIZERS_PARALLELISM=false
    deterministic_algorithms=True
    cudnn_benchmark=False
    cudnn_deterministic=True
    random_seed=0
    torch.cuda.manual_seed_all(0) → 1 GPU(s)
    TDD RED→GREEN: Environment Contract
  ✓ PASS: Every required dependency must be importable and meet version constraints
  ✓ PASS: CUDA GPU must be detected for training
  ✓ PASS: GPU must have at least 10GB VRAM for transformer fine-tuning
  ✓ PASS: PyTorch must be compiled with CUDA support
  ✓ PASS: Cross-dependency version constraints must be satisfied
  
    Preflight Checks — Failure Isolation Gate
  ✓ PASS: GPU count 1 (allocated)
  ✓ PASS: GPU[0] NVIDIA A100-SXM4-80GB | cap (8, 0) | 85.1GB
  ✓ PASS: torch CUDA runtime 11.7
  ✓ PASS: Disk 201.9GB free
  ✓ PASS: src/repro.py importable
  ✓ PASS: repro_cfg['PYTHONHASHSEED'] = '0'
  ✓ PASS: repro_cfg['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
  ✓ PASS: repro

In [ ]:
# GCS Setup and Authentication - Streaming access only (assumes data already in GCS)
import os
import pathlib
import subprocess

# Install gcsfs if not present
try:
    import gcsfs
except ImportError:
    subprocess.run([".venv/bin/pip", "install", "gcsfs"], check=True)
    import gcsfs

# Authenticate with GCS (Colab will prompt for authentication)
from google.colab import auth
auth.authenticate_user()

# Configure GCS bucket
GCS_BUCKET_NAME = os.environ.get("GCS_BUCKET_NAME", "apcomp-209b-bucket")
GCS_PROJECT_ID = os.environ.get("GCS_PROJECT_ID", "your-project-id")

# Initialize gcsfs for streaming
fs = gcsfs.GCSFileSystem(project=GCS_PROJECT_ID)

print(f"GCS bucket: {GCS_BUCKET_NAME}")
print(f"GCS project: {GCS_PROJECT_ID}")

# Helper functions for GCS streaming operations
def gcs_exists(blob_path):
    """Check if a file exists in GCS bucket"""
    full_path = f"gs://{GCS_BUCKET_NAME}/{blob_path}"
    return fs.exists(full_path)

def gcs_list_files(prefix):
    """List all files with given prefix in GCS"""
    full_path = f"gs://{GCS_BUCKET_NAME}/{prefix}"
    files = fs.ls(full_path)
    return [f.replace(f"gs://{GCS_BUCKET_NAME}/", "") for f in files if not f.endswith('/')]

def gcs_open(blob_path, mode='rb'):
    """Open a GCS file for streaming (returns file-like object)"""
    full_path = f"gs://{GCS_BUCKET_NAME}/{blob_path}"
    return fs.open(full_path, mode)

def get_gcs_path(blob_path):
    """Get full GCS path for use with libraries that support gs:// URIs"""
    return f"gs://{GCS_BUCKET_NAME}/{blob_path}"

# Set environment variables for pipeline
os.environ["GCS_BUCKET_NAME"] = GCS_BUCKET_NAME
os.environ["USE_GCS_STREAMING"] = "true"

print("\n✓ GCS streaming ready")
print("All data operations will stream directly from GCS")
print("Assumes all required data is already present in GCS")


Mounted at /content/drive
Drive free: 191.8 GB / total: 259.7 GB
/content/drive/MyDrive/cs1090b_cl_bulk ready


In [ ]:
# Verify CourtListener bulk CSVs are present in GCS
import pathlib

GCS_CL_BULK_PREFIX = "cs1090b_cl_bulk/"

print(f"GCS prefix: {GCS_CL_BULK_PREFIX}")
print("=" * 60)

# Check for required files in GCS
required_files = [
    "courts-2026-03-31.csv.bz2",
    "dockets-2026-03-31.csv.bz2", 
    "opinion-clusters-2026-03-31.csv.bz2",
    "opinions-2026-03-31.csv.bz2"
]

print("Verifying bulk CSVs in GCS:")
all_present = True
for filename in required_files:
    gcs_path = GCS_CL_BULK_PREFIX + filename
    
    if gcs_exists(gcs_path):
        # Get file size
        info = fs.info(get_gcs_path(gcs_path))
        size_gb = info['size'] / 1e9
        print(f"  ✓ {filename}  {size_gb:.2f} GB")
    else:
        print(f"  ✗ {filename} NOT FOUND")
        all_present = False

if not all_present:
    raise RuntimeError("Required bulk CSV files not found in GCS. Please upload them first.")

print(f"\n✓ All {len(required_files)} bulk CSVs present in GCS")

# Set GCS path for pipeline to use
os.environ["CL_BULK_GCS_PREFIX"] = GCS_CL_BULK_PREFIX


already symlinked -> /content/drive/MyDrive/cs1090b_cl_bulk
  courts-2026-03-31.csv.bz2  0.00 GB
  dockets-2026-03-31.csv.bz2  4.98 GB
  opinion-clusters-2026-03-31.csv.bz2  2.45 GB
  opinions-2026-03-31.csv.bz2  54.19 GB


In [ ]:
# Cell 4: Skip - Bulk CSVs already in GCS
# This cell is no longer needed as we assume data is already in GCS
print("✓ Skipping bulk CSV download - data already in GCS")
print(f"  Pipeline will stream from: gs://{GCS_BUCKET_NAME}/{os.environ.get('CL_BULK_GCS_PREFIX', 'cs1090b_cl_bulk/')}")


bulk_dir -> /content/drive/MyDrive/cs1090b_cl_bulk
  All 4 bulk CSVs already present in data/raw/cl_bulk - skipping
    courts-2026-03-31.csv.bz2  0.00 GB
    dockets-2026-03-31.csv.bz2  4.98 GB
    opinion-clusters-2026-03-31.csv.bz2  2.45 GB
    opinions-2026-03-31.csv.bz2  54.19 GB


In [ ]:
# Cell 5: Verify shards are present in GCS (skip pipeline if already processed)
# Assumes shards are already in GCS from previous run
import os
import pathlib

os.chdir("/content/cs1090b_HallucinationLegalRAGChatbots")

GCS_SHARDS_PREFIX = "cs1090b_cl_federal_appellate_bulk/"

print(f"GCS shards prefix: {GCS_SHARDS_PREFIX}")
print("=" * 60)

# Check if shards already exist in GCS
manifest_gcs = GCS_SHARDS_PREFIX + "manifest.json"

if gcs_exists(manifest_gcs):
    print("✓ Manifest found in GCS - shards already processed")
    
    # List available shards in GCS
    shard_files = [f for f in gcs_list_files(GCS_SHARDS_PREFIX) if f.endswith('.jsonl')]
    print(f"  {len(shard_files)} shards available in GCS for streaming")
    
    # Show sample of shards
    for f in sorted(shard_files)[:5]:
        info = fs.info(get_gcs_path(f))
        size_mb = info['size'] / 1e6
        print(f"  {f.split('/')[-1]}  {size_mb:.1f} MB")
    
    if len(shard_files) > 5:
        print(f"  ... and {len(shard_files) - 5} more shards")
    
    print("\n✓ Pipeline skipped - using existing shards from GCS")
else:
    raise RuntimeError(
        f"Shards not found in GCS at gs://{GCS_BUCKET_NAME}/{GCS_SHARDS_PREFIX}\n"
        "Please run the pipeline to generate shards first, or upload pre-processed shards to GCS."
    )

# Set environment variable for downstream cells to stream from GCS
os.environ["CL_SHARDS_GCS_PREFIX"] = GCS_SHARDS_PREFIX
print(f"\nDownstream cells will stream from: gs://{GCS_BUCKET_NAME}/{GCS_SHARDS_PREFIX}")


In [ ]:
# Cell 6: Dataset readiness probe — streams from GCS, no local download
# Uses src.dataset_probe (v2.5.11, 303 contract tests, frozen ProbeConfig).
# Polars can read directly from gs:// URIs for memory-efficient streaming.
import json
import os
import pathlib
import sys

os.chdir("/content/cs1090b_HallucinationLegalRAGChatbots")

from src.dataset_probe import ProbeConfig, run_probe
from src.timer import cell_timer


def _get_cell6_logger():
    lg = logging.getLogger("cell6")
    lg.setLevel(logging.INFO)
    if not lg.handlers:
        h = logging.StreamHandler(sys.stdout)
        h.setFormatter(logging.Formatter("  %(message)s"))
        lg.addHandler(h)
    lg.propagate = False
    return lg


logger = _get_cell6_logger()

# Use GCS path for streaming (Polars supports gs:// URIs with gcsfs)
GCS_SHARDS_PREFIX = os.environ.get("CL_SHARDS_GCS_PREFIX", "cs1090b_cl_federal_appellate_bulk/")
gcs_shard_path = f"gs://{GCS_BUCKET_NAME}/{GCS_SHARDS_PREFIX}"

out_path = pathlib.Path("logs/dataset_probe_report.json")
out_path.parent.mkdir(parents=True, exist_ok=True)

with cell_timer("Cell 6 - Dataset readiness probe (streaming from GCS)", logger=logger):
    logger.info("=" * 60)
    logger.info("  run_probe (streaming from GCS via Polars scan_ndjson)")
    logger.info("=" * 60)
    logger.info("  GCS path: %s" % gcs_shard_path)

    probe_cfg = ProbeConfig()
    
    # Note: dataset_probe may need modification to accept gs:// paths
    # For now, we'll use the local path but note this should stream
    report = run_probe(
        data_dir=gcs_shard_path,  # Polars can handle gs:// URIs
        subset=1_465_484,  # full corpus
        full_scan=True,
        cfg=probe_cfg,
    )

    out_path.write_text(json.dumps(report.model_dump(), indent=2, default=str))
    logger.info("  report -> %s" % out_path)

    logger.info("\n" + "=" * 60)
    logger.info("  Probe summary")
    logger.info("=" * 60)
    summary = report.summary
    logger.info("  all_passed:     %s" % summary.get("all_passed"))
    logger.info("  passed_count:   %d" % summary.get("passed_count", 0))
    logger.info("  failed_block:   %d" % summary.get("failed_blocking_count", 0))
    logger.info("  subset_n:       %s" % format(report["subset_n"], ","))
    logger.info("  parse_errors:   %d" % report.get("parse_errors", 0))

    logger.info("\n  Gate results:")
    for gate_name, gate_result in report["gates"].items():
        status = "PASS" if gate_result.get("pass") else "FAIL"
        sev = gate_result.get("severity", "?")
        logger.info("    %-10s %s  (%s)" % (gate_name, status, sev))

    prov = report.get("provenance", {})
    logger.info("\n  probe_version: %s" % prov.get("probe_version"))
    logger.info("  polars_version: %s" % prov.get("polars_version"))
    logger.info("  full_scan: %s" % prov.get("full_scan"))
    logger.info("  streaming: GCS (no local download)")

    if not summary.get("all_passed"):
        raise RuntimeError("probe gates failed — see report")
    logger.info("\nOK all gates passed — corpus cleared for Stage 3")


In [ ]:
# Cell 7: Verify LePaRD dataset is present in GCS
# Assumes LePaRD data is already uploaded to GCS
import os
import pathlib

os.chdir("/content/cs1090b_HallucinationLegalRAGChatbots")

GCS_LEPARD_PREFIX = "cs1090b_lepard/"

print(f"GCS LePaRD prefix: {GCS_LEPARD_PREFIX}")
print("=" * 60)

# Check if LePaRD files exist in GCS
lepard_jsonl_gcs = GCS_LEPARD_PREFIX + "lepard_train_4000000_rev0194f95.jsonl"

if gcs_exists(lepard_jsonl_gcs):
    print("✓ LePaRD JSONL found in GCS")
    
    # List available files in GCS
    lepard_files = gcs_list_files(GCS_LEPARD_PREFIX)
    for f in lepard_files:
        info = fs.info(get_gcs_path(f))
        size = info['size']
        size_gb = size / 1e9 if size > 1e6 else size
        unit = "GB" if size > 1e6 else "B"
        print(f"  {f.split('/')[-1]}  {size_gb:.2f} {unit}")
    
    print("\n✓ LePaRD ready for streaming access")
else:
    raise RuntimeError(
        f"LePaRD dataset not found in GCS at gs://{GCS_BUCKET_NAME}/{lepard_jsonl_gcs}\n"
        "Please upload the LePaRD dataset to GCS first."
    )

# Set environment variable for training cells to stream from GCS
os.environ["LEPARD_GCS_PREFIX"] = GCS_LEPARD_PREFIX
lepard_gcs_path = f"gs://{GCS_BUCKET_NAME}/{lepard_jsonl_gcs}"
print(f"Training will stream from: {lepard_gcs_path}")


In [ ]:
# Cell 8: LePaRD ↔ CourtListener compatibility audit
# Uses src.lepard_cl_compat (56 TDD tests) — pure analysis, deterministic,
# CI-gate capable. Reports usable gold pair rate (both endpoints in CL corpus).
import os
import pathlib
import subprocess
import sys

os.chdir("/content/cs1090b_HallucinationLegalRAGChatbots")


def _stream(cmd):
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
        text=True,
        env=env,
    )
    for line in proc.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
    proc.wait()
    return proc.returncode


print("=" * 60)
print("  src.lepard_cl_compat — pair-level compatibility audit")
print("=" * 60)

rc = _stream([".venv/bin/python", "-m", "src.lepard_cl_compat"])
if rc != 0:
    raise SystemExit(f"compat audit failed with exit code {rc}")

print("\nOK compatibility audit complete")

In [ ]:
# Cell 9: Data quality gate — streams from GCS, no local download
# Uses scripts/audit_jsonl_nan.py — typed counters, W&B telemetry, CI gate.
# Audit script will stream shards directly from GCS using gs:// URIs.
import json
import os
import pathlib
import subprocess
import sys

os.chdir("/content/cs1090b_HallucinationLegalRAGChatbots")

GCS_SHARDS_PREFIX = os.environ.get("CL_SHARDS_GCS_PREFIX", "cs1090b_cl_federal_appellate_bulk/")
gcs_shard_path = f"gs://{GCS_BUCKET_NAME}/{GCS_SHARDS_PREFIX}"

print(f"Streaming shards from: {gcs_shard_path}")


def _stream(cmd):
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
        text=True,
        env=env,
    )
    lines = []
    for line in proc.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
        lines.append(line)
    proc.wait()
    return proc.returncode, "".join(lines)


print("=" * 60)
print("  scripts/audit_jsonl_nan.py --json (streaming from GCS)")
print("=" * 60)

# Note: audit script may need modification to accept gs:// paths
# For now, pass the GCS path and let gcsfs handle it
rc, output = _stream(
    [
        ".venv/bin/python",
        "scripts/audit_jsonl_nan.py",
        "--input-dir",
        gcs_shard_path,
        "--json",
    ]
)
if rc != 0:
    raise SystemExit(f"audit failed with exit code {rc}")

# Parse trailing JSON report
json_start = output.rfind("{")
if json_start >= 0:
    report = json.loads(output[json_start:])
    print("\n" + "=" * 60)
    print("  Audit summary (streamed from GCS)")
    print("=" * 60)
    print(f"  verdict:       {report.get('gate_verdict')}")
    print(f"  clean_pct:     {report.get('clean_pct')}")
    print(f"  total_lines:   {report.get('total_lines'):,}")
    print(f"  total_shards:  {report.get('total_shards')}")
    print(f"  nan_lines:     {report.get('nan_lines')}")
    print(f"  nonfinite:     {report.get('nonfinite_lines')}")
    print(f"  decode_errors: {report.get('decode_error_lines')}")

    if report.get("gate_verdict") != "CLEAN":
        raise RuntimeError(f"audit gate failed: {report.get('gate_verdict')}")

print("\nOK all shards pass data quality gate (streamed from GCS)")


## Pipeline Summary — Stages 1–3 + Readiness Gates

End-to-end CourtListener + LePaRD data pipeline for the Legal RAG project. Every cell is thin orchestration over TDD-covered modules in `src/` and `scripts/` — no business logic lives in the notebook.

### Cell map

| Cell | Stage | Purpose | Modules / Scripts |
|---|---|---|---|
| **1** | — | Title and scope | (markdown) |
| **2** | Bootstrap | Clone repo, install `uv`, create `.venv` (Python 3.11.9), sync pinned deps, write `.env` | `uv`, `pyproject.toml`, `uv.lock` |
| **3** | Preflight | Reproducibility config + GPU/dependency contract + preflight gate | `src.repro`, `src.environment`, `src.timer` |
| **4** | Stage 1 (acq) | Discover and download 4 CourtListener bulk CSVs from public S3 (~62 GB), persist to Google Drive, skip if present | `src.config`, `src.s3_discovery`, `src.bulk_download` |
| **5** | Stages 1–2 | Filter chain (courts → dockets → clusters), extract 1.46M opinions to 159 JSONL shards, write manifest with SHA-256 checksums, run TDD contract tests | `src.pipeline` (`run_pipeline`, `validate_pipeline`), which chains `src.manifest`, `src.s3_discovery`, `src.bulk_download`, `src.filter_chain`, `src.extract`, `src.validation`, `src.schemas`, `src.exceptions` |
| **6** | Stage 3 readiness | Full-corpus Polars `scan_ndjson` + 8 gates (schema, A7, A8, A9, A11, A12, A13, B6) | `src.dataset_probe` (v2.5.11, 303 contract tests) |
| **7** | Stage 1 (LePaRD) | Ingest LePaRD 4M training pairs from Hugging Face at pinned revision `0194f95c…`, with sidecar + manifest; self-heals missing sidecars from disk bytes | `scripts/ingest_lepard.py` (79 TDD tests) |
| **8** | Readiness | LePaRD ↔ CourtListener compatibility audit — reports usable gold pair rate (both endpoints present in CL corpus) | `src.lepard_cl_compat` (56 TDD tests) |
| **9** | Data quality gate | NaN / encoding / parse audit over all shards; verdict: `CLEAN / REPAIRABLE / HARD_FAILURE / PARSE_FAILURE` | `scripts/audit_jsonl_nan.py` |

### Persistence strategy

Google Colab ephemeral storage is replaced with Google Drive symlinks so no stage repeats across sessions:

```
data/raw/cl_bulk                   -> /content/drive/MyDrive/cs1090b_cl_bulk
data/raw/cl_federal_appellate_bulk -> /content/drive/MyDrive/cs1090b_cl_federal_appellate_bulk
data/raw/lepard                    -> /content/drive/MyDrive/cs1090b_lepard
```

Fast-path behavior on reconnect:
- Cell 4: `download_file` skips any file already on Drive → ~0s
- Cell 5: `run_pipeline` reads manifest, validates shard SHA-256, returns immediately → ~1s
- Cell 6: Polars memory-mapped scan → ~30s regardless
- Cell 7: `--verify-only` checks sidecar + manifest + digest → ~5s
- Cell 9: 48-core parallel audit → ~30s

### Reproducibility guarantees

- **Python 3.11.9** pinned via `.python-version`
- **uv.lock** pins every dependency transitively (`torch 2.0.1+cu117`, `transformers 4.41.2`, `numpy 1.26.4`, …)
- **`src.repro.configure()`** sets `PYTHONHASHSEED=0`, `CUBLAS_WORKSPACE_CONFIG=:4096:8`, `torch.use_deterministic_algorithms(True)`, `cudnn.benchmark=False`, seeds Python/NumPy/torch CPU/CUDA RNGs to 0
- **LePaRD revision pinned** to `0194f95c3091acceab3b887c9b09ef432cf84052` (40-char SHA; mutable refs rejected)
- **Manifest checksums** — every shard's SHA-256 recorded; validation runs on every pipeline invocation
- **Contract tests** — `validate_pipeline` runs 13 `check_*` contract tests over sampled shards after extraction

### Downstream (not in this notebook)

README stages 4–7 — index generation (BM25 + FAISS), encoder training (BGE-M3 + Legal-BERT), sequential-loading evaluation (Tier A/B/C), W&B experiment tracking — are **not started** per the README and are correctly excluded from the data-pipeline notebook. They will be driven by `src.dataset_loader`, `src.lightning_datamodule`, `src.model_loader`, `src.split`, `src.wandb_logger`, and `src.hf_export` when that work begins.

### Verdict

> The full 1,465,484-record CourtListener Federal Appellate corpus plus the 4M-pair LePaRD training set are acquired, audited, probed, and verified compatible. The data pipeline is complete and reproducible across Colab sessions via Google Drive persistence. Ready for Stage 4 (indexing) whenever training begins.